<a href="https://colab.research.google.com/github/wahyuwinanta/WeldingApp/blob/main/notebooks/yolov8n-instance-segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

In [ ]:
import os
import shutil
import random
import glob
from ultralytics import YOLO
from IPython.display import display, Image

# Setup direktori kerja
HOME = os.getcwd()
!pip install ultralytics roboflow --quiet

In [ ]:
# 1. Download Dataset dari Roboflow (Pastikan ganti API Key & Project Anda)
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY") # Ganti dengan API Key Anda
project = rf.workspace("model-examples").project("car-parts-instance-segmentation")
dataset = project.version(3).download("yolov8")

In [ ]:
# 2. Fungsi Splitting Dataset
def split_dataset(source_path, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1):
    # Path folder tujuan
    base_dir = os.path.join(HOME, 'split_dataset')
    for split in ['train', 'valid', 'test']:
        for sub in ['images', 'labels']:
            os.makedirs(os.path.join(base_dir, split, sub), exist_ok=True)

    # Ambil semua file gambar dari folder train bawaan Roboflow
    image_files = glob.glob(os.path.join(source_path, 'train/images/*.jpg'))
    random.shuffle(image_files)

    total = len(image_files)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    splits = {
        'train': image_files[:train_end],
        'valid': image_files[train_end:val_end],
        'test': image_files[val_end:]
    }

    for split_name, files in splits.items():
        for img_path in files:
            # Pindahkan Gambar
            fname = os.path.basename(img_path)
            shutil.copy(img_path, os.path.join(base_dir, split_name, 'images', fname))

            # Pindahkan Label (TXT) yang bersesuaian
            label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt')
            if os.path.exists(label_path):
                shutil.copy(label_path, os.path.join(base_dir, split_name, 'labels', fname.replace('.jpg', '.txt')))

    return base_dir

# Eksekusi splitting
new_dataset_path = split_dataset(dataset.location)

In [ ]:
# 3. Update file data.yaml agar mengarah ke folder baru
import yaml
with open(f"{dataset.location}/data.yaml", 'r') as f:
    data_yaml = yaml.safe_load(f)

data_yaml['train'] = os.path.join(new_dataset_path, 'train/images')
data_yaml['val'] = os.path.join(new_dataset_path, 'valid/images')
data_yaml['test'] = os.path.join(new_dataset_path, 'test/images')

with open(f"{new_dataset_path}/data.yaml", 'w') as f:
    yaml.dump(data_yaml, f)

print(f"Dataset berhasil dibagi! Lokasi: {new_dataset_path}")

In [ ]:
# Inisialisasi model pre-trained
model = YOLO('yolov8n-seg.pt')

# Mulai Training
results = model.train(
    data=f"{new_dataset_path}/data.yaml",
    epochs=100,
    imgsz=640,
    device=0, # Gunakan 0 untuk GPU, atau 'cpu'
    name='yolov8_custom_seg',
    exist_ok=True
)

In [ ]:
import pandas as pd

# 1. Mengambil hasil validasi
metrics = model.val()

# 2. Menghitung Kecepatan (Inference Time & FPS)
# YOLOv8 menyimpan waktu dalam ms: [preprocess, inference, loss, postprocess]
t_preprocess = metrics.speed['preprocess']
t_inference = metrics.speed['inference']
t_postprocess = metrics.speed['postprocess']
total_time_ms = t_preprocess + t_inference + t_postprocess
fps = 1000 / total_time_ms

# 3. Membuat Tabel Ringkasan untuk Laporan Skripsi
summary_data = {
    "Matriks Evaluasi": ["mAP@50 (B)", "mAP@50-95 (B)", "mAP@50 (M)", "mAP@50-95 (M)",
                         "Precision", "Recall", "Inference Time", "Kecepatan (FPS)"],
    "Nilai": [
        f"{metrics.box.map50:.4f}",
        f"{metrics.box.map:.4f}",
        f"{metrics.seg.map50:.4f}",
        f"{metrics.seg.map:.4f}",
        f"{metrics.box.mp:.4f}",
        f"{metrics.box.mr:.4f}",
        f"{total_time_ms:.2f} ms/img",
        f"{fps:.2f} FPS"
    ]
}

df_summary = pd.DataFrame(summary_data)
print("\n=== TABEL EVALUASI MODEL UNTUK SKRIPSI ===")
display(df_summary)

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Path folder hasil training
RESULT_DIR = f"{HOME}/runs/segment/yolov8_custom_seg"

def show_results(image_name, title):
    img_path = os.path.join(RESULT_DIR, image_name)
    if os.path.exists(img_path):
        plt.figure(figsize=(10, 8))
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.imshow(img)
        plt.title(title)
        plt.axis('off')
        plt.show()
    else:
        print(f"Grafik {image_name} tidak ditemukan.")

# Menampilkan grafik evaluasi utama
show_results("confusion_matrix.png", "Confusion Matrix")
show_results("F1_curve.png", "F1-Score Curve")
show_results("results.png", "Grafik Training Loss & Accuracy") # Berisi Box Loss, Class Loss, DFL
show_results("PR_curve.png", "Precision-Recall Curve")

In [ ]:
# Membaca file results.csv yang dihasilkan saat training
results_csv_path = os.path.join(RESULT_DIR, 'results.csv')

if os.path.exists(results_csv_path):
    results_df = pd.read_csv(results_csv_path)
    results_df.columns = results_df.columns.str.strip() # Bersihkan spasi pada nama kolom

    # Ambil baris terakhir (hasil akhir training)
    last_loss = results_df.iloc[-1]

    print("\n=== DETAIL LOSS FUNCTION (FINAL EPOCH) ===")
    print(f"Train Box Loss  : {last_loss['train/box_loss']:.4f}")
    print(f"Train Class Loss: {last_loss['train/cls_loss']:.4f}")
    print(f"Train DFL Loss  : {last_loss['train/dfl_loss']:.4f}")
    print("-" * 40)
    print(f"Val Box Loss    : {last_loss['val/box_loss']:.4f}")
    print(f"Val Class Loss  : {last_loss['val/cls_loss']:.4f}")
    print(f"Val DFL Loss    : {last_loss['val/dfl_loss']:.4f}")

In [ ]:
# Ekspor ke TFLite (Float16 untuk optimasi speed & size)
export_path = model.export(format="tflite", half=True)

print(f"--- PROSES SELESAI ---")
print(f"Model terbaik tersimpan di: {HOME}/runs/segment/yolov8_custom_seg/weights/best.pt")
print(f"Model TFLite tersimpan di: {export_path}")